In [9]:
import numpy as np

sentence = " 远方有棵苹果树"
L = 7  # 序列长度
d_model = 4     # 词向量维度

# 模拟embedding后的输入
np.random.seed(42)
X = np.random.rand(L, d_model)  # (L, d_model)

print("Input X shape:", X.shape)
print("Input X:\n", X)



Input X shape: (7, 4)
Input X:
 [[0.37454012 0.95071431 0.73199394 0.59865848]
 [0.15601864 0.15599452 0.05808361 0.86617615]
 [0.60111501 0.70807258 0.02058449 0.96990985]
 [0.83244264 0.21233911 0.18182497 0.18340451]
 [0.30424224 0.52475643 0.43194502 0.29122914]
 [0.61185289 0.13949386 0.29214465 0.36636184]
 [0.45606998 0.78517596 0.19967378 0.51423444]]


# 1.计算Q、K、V

In [10]:
# 定义三个权重矩阵 W_q, W_k, W_v
d_k = 4     # 查询Q和键K的维度
d_v = 4     # 值V的维度

W_q = np.random.rand(d_model, d_k)  # (d_model, d_k)
W_k = np.random.rand(d_model, d_k)  # (d_model, d_k)
W_v = np.random.rand(d_model, d_v)  # (d_model, d_v)

print("\nWeight matrices:")
print("W_q shape:", W_q.shape)
print("W_k shape:", W_k.shape)
print("W_v shape:", W_v.shape)  

# 计算查询Q、键K和值V
Q = np.dot(X, W_q)  # (L, d_k)
K = np.dot(X, W_k)  # (L, d_k)
V = np.dot(X, W_v)  # (L, d_v)

print("\nComputed Q, K, V:")
print("Q shape:", Q.shape)
print("K shape:", K.shape)
print("V shape:", V.shape)


Weight matrices:
W_q shape: (4, 4)
W_k shape: (4, 4)
W_v shape: (4, 4)

Computed Q, K, V:
Q shape: (7, 4)
K shape: (7, 4)
V shape: (7, 4)


# 2.计算注意力分数

In [11]:
# 计算注意力分数矩阵
# Q @ K^T: (7, 4) @ (4, 7) = (7, 7)
attention_scores = np.dot(Q, K.T) 

print("\n注意力分数矩阵shape：",attention_scores.shape)
print("注意力分数矩阵:\n", attention_scores)


注意力分数矩阵shape： (7, 7)
注意力分数矩阵:
 [[7.89340388 1.97403678 4.90772439 3.6891315  4.67384994 3.54464393
  5.16595985]
 [3.22565907 0.87211967 2.0296429  1.56291992 1.9153765  1.52318711
  2.0908536 ]
 [6.59796129 1.69540136 4.10580755 3.13641856 3.91331537 3.03186162
  4.29180994]
 [3.4853352  0.80518594 2.08896562 1.53080736 2.05420949 1.4750693
  2.2609634 ]
 [4.52936604 1.11853981 2.80328496 2.09782145 2.68001849 2.01483472
  2.96302725]
 [3.43279384 0.82263223 2.08131814 1.53324272 2.02518399 1.48193846
  2.22845918]
 [5.92061565 1.48959494 3.67852585 2.79057752 3.50915505 2.68458589
  3.86566699]]


# 3.缩放（Scaled）

In [14]:
# 缩放：除以 sqrt(d_k)
scaled_attention_scales = attention_scores / np.sqrt(d_k)

print("\n缩放后的注意力分数矩阵shape：",scaled_attention_scales.shape)
print("缩放后的注意力分数矩阵:\n", scaled_attention_scales)
print("\n缩放前的值范围：", np.min(attention_scores), np.max(attention_scores))
print("缩放后的值范围：", np.min(scaled_attention_scales), np.max(scaled_attention_scales))


缩放后的注意力分数矩阵shape： (7, 7)
缩放后的注意力分数矩阵:
 [[3.94670194 0.98701839 2.45386219 1.84456575 2.33692497 1.77232197
  2.58297993]
 [1.61282953 0.43605983 1.01482145 0.78145996 0.95768825 0.76159355
  1.0454268 ]
 [3.29898065 0.84770068 2.05290378 1.56820928 1.95665768 1.51593081
  2.14590497]
 [1.7426676  0.40259297 1.04448281 0.76540368 1.02710474 0.73753465
  1.1304817 ]
 [2.26468302 0.5592699  1.40164248 1.04891073 1.34000924 1.00741736
  1.48151362]
 [1.71639692 0.41131611 1.04065907 0.76662136 1.012592   0.74096923
  1.11422959]
 [2.96030783 0.74479747 1.83926292 1.39528876 1.75457752 1.34229294
  1.93283349]]

缩放前的值范围： 0.8051859417190418 7.893403878529283
缩放后的值范围： 0.4025929708595209 3.9467019392646416


# 4.softmax归一化

In [15]:
def softmax(x):
    '''对每一行做softmax'''
    exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))    # 减去最大值，防止溢出
    return exp_x / np.sum(exp_x,axis=1,keepdims=True)

# 对每一行做softmax，得到注意力权重矩阵
attention_weights = softmax(scaled_attention_scales)

print("\n注意力权重矩阵shape：",attention_weights.shape)
print("第一个token的注意力权重：", attention_weights[0,:])
print("每一行的和：", np.sum(attention_weights, axis=1))  # 每一行的和应该为1



注意力权重矩阵shape： (7, 7)
第一个token的注意力权重： [0.50810889 0.02633799 0.11418912 0.06208843 0.10158733 0.05776112
 0.12992713]
每一行的和： [1. 1. 1. 1. 1. 1. 1.]


# 5.加权求和

In [17]:
# 用注意力权重加权V
# （ 7,7） @ （7,4） = （7,4）
output = np.dot(attention_weights,V)

print("\n输出矩阵shape：",output.shape)
print("输入形状：", X.shape)
print("Input X:\n", X)
print("输出矩阵:\n", output)


输出矩阵shape： (7, 4)
输入形状： (7, 4)
Input X:
 [[0.37454012 0.95071431 0.73199394 0.59865848]
 [0.15601864 0.15599452 0.05808361 0.86617615]
 [0.60111501 0.70807258 0.02058449 0.96990985]
 [0.83244264 0.21233911 0.18182497 0.18340451]
 [0.30424224 0.52475643 0.43194502 0.29122914]
 [0.61185289 0.13949386 0.29214465 0.36636184]
 [0.45606998 0.78517596 0.19967378 0.51423444]]
输出矩阵:
 [[0.4187476  1.45325569 1.23624383 1.26164869]
 [0.3782628  1.23256461 1.12412848 1.10981887]
 [0.40818322 1.38992818 1.20443925 1.21911573]
 [0.38226027 1.25376307 1.13292744 1.12537603]
 [0.39128172 1.29669672 1.15604096 1.15636705]
 [0.38145372 1.24964218 1.13119128 1.1225215 ]
 [0.40299534 1.35963583 1.18865996 1.1992695 ]]
